In [4]:
# ═══════════════════════════════════════════════
#  CELL 1 — FULL SETUP (ran after reconnect)
# ═══════════════════════════════════════════════
!pip install kagglehub timm xgboost thop -q

from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, copy, json, shutil, warnings
import numpy as np
import pandas as pd
from PIL import Image
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, roc_auc_score)
from sklearn.preprocessing  import label_binarize
from sklearn.linear_model   import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.svm             import SVC
from xgboost                 import XGBClassifier

try:
    from thop import profile as thop_profile
    THOP_AVAILABLE = True
except: THOP_AVAILABLE = False

# ── Config ─────────────────────────────────────
DRIVE_DIR = '/content/drive/MyDrive/computer vison/HAM10000_Results'
SAVE_DIR   = './results'
IMG_SIZE   = 224
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,  exist_ok=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# ── Download dataset ────────────────────────────
import kagglehub
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

csv_path = None
for root, dirs, files in os.walk(path):
    for f in files:
        if 'metadata' in f.lower() and f.endswith('.csv'):
            csv_path = os.path.join(root, f); break
    if csv_path: break

img_map = {}
for root, dirs, files in os.walk(path):
    for f in files:
        if f.lower().endswith('.jpg'):
            img_map[os.path.splitext(f)[0]] = os.path.join(root, f)
print(f"Images: {len(img_map)}")

df          = pd.read_csv(csv_path)
df          = df.drop_duplicates(subset='lesion_id', keep='first')
df          = df[df['image_id'].isin(img_map)].reset_index(drop=True)
df['path']  = df['image_id'].map(img_map)
CLASSES     = sorted(df['dx'].unique())
NUM_CLASSES = len(CLASSES)
cls2idx     = {c:i for i,c in enumerate(CLASSES)}
df['label'] = df['dx'].map(cls2idx)

train_df, test_df = train_test_split(df, test_size=0.30,
                    stratify=df['label'], random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.214,
                    stratify=train_df['label'], random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f"Train:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}")
print(f"Classes({NUM_CLASSES}): {CLASSES}")

# ── Transforms ─────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(0.3,0.3,0.3,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ── Disk Dataset ────────────────────────────────
class DiskDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, int(row['label'])

def make_loaders(bs):
    labels       = train_df['label'].values
    class_counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    weights      = 1.0 / class_counts[labels]
    sampler      = WeightedRandomSampler(weights, len(weights))
    tr = DataLoader(DiskDataset(train_df,train_tf), batch_size=bs,
                    sampler=sampler, num_workers=4, pin_memory=True)
    vl = DataLoader(DiskDataset(val_df,  val_tf),  batch_size=bs,
                    shuffle=False,  num_workers=4, pin_memory=True)
    te = DataLoader(DiskDataset(test_df, val_tf),  batch_size=bs,
                    shuffle=False,  num_workers=4, pin_memory=True)
    return tr, vl, te

# ── Model factory ───────────────────────────────
def get_model(name):
    n = name.lower().replace('-','').replace('_','')
    if n == 'alexnet':
        m = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.classifier.parameters(): p.requires_grad = True
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    elif n == 'vgg16':
        m = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.classifier.parameters(): p.requires_grad = True
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    elif n == 'vgg19':
        m = models.vgg19(weights=models.VGG19_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.classifier.parameters(): p.requires_grad = True
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    elif n == 'resnet18':
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.layer3.parameters(): p.requires_grad = True
        for p in m.layer4.parameters(): p.requires_grad = True
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif n == 'resnet50':
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.layer3.parameters(): p.requires_grad = True
        for p in m.layer4.parameters(): p.requires_grad = True
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif n == 'resnet101':
        m = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.layer4.parameters(): p.requires_grad = True
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif n == 'densenet121':
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.features.denseblock4.parameters(): p.requires_grad = True
        m.classifier = nn.Linear(m.classifier.in_features, NUM_CLASSES)
    elif n == 'efficientnetb0':
        m = timm.create_model('efficientnet_b0', pretrained=True,
                               num_classes=NUM_CLASSES)
        for p in m.parameters(): p.requires_grad = False
        for p in m.blocks[-3].parameters(): p.requires_grad = True
        for p in m.blocks[-2].parameters(): p.requires_grad = True
        for p in m.blocks[-1].parameters(): p.requires_grad = True
        for p in m.classifier.parameters(): p.requires_grad = True
    else: raise ValueError(name)
    tr  = sum(p.numel() for p in m.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in m.parameters())
    print(f"  {name:<18} trainable:{tr:>10,}/{tot:>10,} ({100*tr/tot:.1f}%)")
    return m

MODEL_NAMES = ['AlexNet','VGG16','VGG19',
               'ResNet18','ResNet50','ResNet101',
               'DenseNet121','EfficientNet-B0']

# ── Helpers ─────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer, scaler=None):
    model.train()
    tl=cr=tot=0
    for imgs,labels in loader:
        imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
        optimizer.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast():
                out=model(imgs); loss=criterion(out,labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        else:
            out=model(imgs); loss=criterion(out,labels)
            loss.backward(); optimizer.step()
        tl+=loss.item()*imgs.size(0)
        cr+=(out.argmax(1)==labels).sum().item()
        tot+=imgs.size(0)
    return tl/tot, cr/tot

def evaluate(model, loader):
    model.eval()
    ap,al,ab=[],[],[]
    with torch.no_grad():
        for imgs,labels in loader:
            out=model(imgs.to(DEVICE))
            ab.extend(torch.softmax(out,1).cpu().numpy())
            ap.extend(out.argmax(1).cpu().numpy())
            al.extend(labels.numpy())
    yt=np.array(al); yp=np.array(ap); yb=np.array(ab)
    acc =accuracy_score(yt,yp)*100
    prec=precision_score(yt,yp,average='weighted',zero_division=0)*100
    rec =recall_score(yt,yp,   average='weighted',zero_division=0)*100
    f1  =f1_score(yt,yp,        average='weighted',zero_division=0)*100
    try:
        ybin=label_binarize(yt,classes=list(range(NUM_CLASSES)))
        auc =roc_auc_score(ybin,yb,multi_class='ovr',average='weighted')*100
    except: auc=0.0
    return round(acc,2),round(prec,2),round(rec,2),round(f1,2),round(auc,2)

def compute_efficiency(model):
    dummy=torch.randn(1,3,IMG_SIZE,IMG_SIZE).to(DEVICE)
    model.eval()
    pm=sum(p.numel() for p in model.parameters())/1e6
    sm=sum(p.numel()*p.element_size() for p in model.parameters())/(1024**2)
    fg='N/A'
    if THOP_AVAILABLE:
        try:
            fl,_=thop_profile(model,inputs=(dummy,),verbose=False)
            fg=round(fl/1e9,2)
        except: pass
    with torch.no_grad():
        for _ in range(5): model(dummy)
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        t0=time.perf_counter()
        for _ in range(50): model(dummy)
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        im=(time.perf_counter()-t0)/50*1000
    return round(pm,2),round(sm,2),fg,round(im,2)

def save_to_drive(filename):
    try:
        shutil.copy2(os.path.join(SAVE_DIR,filename),
                     os.path.join(DRIVE_DIR,filename))
        print(f"  ☁️  {filename}")
    except Exception as e:
        print(f"  ⚠️ Drive save failed: {e}")

print("\n✅ SETUP COMPLETE — Ready to train!")
print(f"Drive folder: {DRIVE_DIR}")
print(f"Files in Drive: {os.listdir(DRIVE_DIR)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4
Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Images: 10015
Train:4109 Val:1120 Test:2241
Classes(7): ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

✅ SETUP COMPLETE — Ready to train!
Drive folder: /content/drive/MyDrive/computer vison/HAM10000_Results
Files in Drive: ['AlexNet_best.pth', 'VGG16_best.pth', 'VGG19_best.pth', 'ResNet18_best.pth', 'ResNet50_best.pth', 'ResNet101_best.pth', 'DenseNet121_best.pth', 'table2.csv', 'all_results.json', 'table1.csv', 'table3.csv']


In [5]:
# ═══════════════════════════════════════════════
#  CELL 2 — TRAIN REMAINING + TABLE 2 + PRINT ALL
# ═══════════════════════════════════════════════

EPOCHS_P1 = 4;  LR_P1 = 1e-3
EPOCHS_P2 = 6;  LR_P2 = 1e-5

BATCH_MAP = {
    'AlexNet':32, 'VGG16':16, 'VGG19':16,
    'ResNet18':32, 'ResNet50':32, 'ResNet101':8,
    'DenseNet121':16, 'EfficientNet-B0':32,
}

# ── Load existing results from Drive ───────────
results_json_local = os.path.join(SAVE_DIR,'all_results.json')
results_json_drive = os.path.join(DRIVE_DIR,'all_results.json')

if os.path.exists(results_json_drive):
    shutil.copy2(results_json_drive, results_json_local)
    with open(results_json_local) as f: saved=json.load(f)
    table1=saved.get('table1',{})
    table3=saved.get('table3',{})
    print(f"✅ Loaded from Drive. Done: {list(table1.keys())}")
else:
    table1,table3={},{}
    print("🆕 No previous results found — starting fresh")

# ── Restore + evaluate any .pth without results ─
for mname in MODEL_NAMES:
    ckpt_name  = mname.replace('-','_')+'_best.pth'
    ckpt_local = os.path.join(SAVE_DIR, ckpt_name)
    ckpt_drive = os.path.join(DRIVE_DIR,ckpt_name)
    if os.path.exists(ckpt_drive) and not os.path.exists(ckpt_local):
        shutil.copy2(ckpt_drive,ckpt_local)
        print(f"  📥 Restored {ckpt_name}")
    if mname not in table1 and os.path.exists(ckpt_local):
        print(f"  📊 Evaluating {mname} from checkpoint...")
        bs    = BATCH_MAP.get(mname,16)
        model = get_model(mname)
        model.load_state_dict(torch.load(ckpt_local,map_location=DEVICE))
        model = model.to(DEVICE)
        _,_,te_l = make_loaders(bs)
        acc,prec,rec,f1,auc = evaluate(model,te_l)
        p,s,fl,im = compute_efficiency(model)
        table1[mname]={'Accuracy (%)':acc,'Precision (%)':prec,
                       'Recall (%)':rec,  'F1-Score (%)':f1,'AUC (%)':auc}
        table3[mname]={'Parameters (M)':p,'Model Size (MB)':s,
                       'FLOPs (G)':fl,    'Inference Time (ms)':im,'Accuracy (%)':acc}
        print(f"     Acc:{acc} F1:{f1} AUC:{auc}")
        del model,te_l; torch.cuda.empty_cache(); gc.collect()

best_backbone = max(table1,key=lambda m:table1[m]['Accuracy (%)']) if table1 else None
best_acc      = table1[best_backbone]['Accuracy (%)'] if best_backbone else 0.0
print(f"\nStatus: {len(table1)}/8 done | Best: {best_backbone} ({best_acc}%)")
print(f"Still need: {[m for m in MODEL_NAMES if m not in table1]}")

# ── Train missing models ────────────────────────
for mname in MODEL_NAMES:
    if mname in table1:
        print(f"⏭️  {mname:<18} Acc:{table1[mname]['Accuracy (%)']}%")
        continue

    bs = BATCH_MAP.get(mname,16)
    print(f"\n{'='*55}\n  {mname}  batch={bs}\n{'='*55}")
    torch.cuda.empty_cache(); gc.collect()

    tr_l,vl_l,te_l = make_loaders(bs)
    model     = get_model(mname).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler    = torch.cuda.amp.GradScaler()
    best_val,best_w = 0.0,None

    # Phase 1
    print(f"\n  📌 Phase 1 ({EPOCHS_P1} ep | LR={LR_P1})")
    opt1 = optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),
                      lr=LR_P1,weight_decay=1e-4)
    sch1 = optim.lr_scheduler.CosineAnnealingLR(opt1,T_max=EPOCHS_P1,eta_min=1e-5)
    for ep in range(1,EPOCHS_P1+1):
        t0=time.time()
        tl,ta=train_epoch(model,tr_l,criterion,opt1,scaler)
        va,*_=evaluate(model,vl_l); sch1.step()
        print(f"  Ep{ep}/{EPOCHS_P1} Loss:{tl:.4f} "
              f"Train:{ta*100:.1f}% Val:{va:.1f}% {time.time()-t0:.0f}s")
        if va>best_val: best_val=va; best_w=copy.deepcopy(model.state_dict())

    # Phase 2
    print(f"\n  🔓 Phase 2 ({EPOCHS_P2} ep | LR={LR_P2})")
    for p in model.parameters(): p.requires_grad=True
    torch.cuda.empty_cache()
    opt2=optim.Adam(model.parameters(),lr=LR_P2,weight_decay=1e-4)
    sch2=optim.lr_scheduler.CosineAnnealingLR(opt2,T_max=EPOCHS_P2,eta_min=1e-7)
    for ep in range(1,EPOCHS_P2+1):
        t0=time.time()
        tl,ta=train_epoch(model,tr_l,criterion,opt2,scaler)
        va,*_=evaluate(model,vl_l); sch2.step()
        print(f"  Ep{ep}/{EPOCHS_P2} Loss:{tl:.4f} "
              f"Train:{ta*100:.1f}% Val:{va:.1f}% {time.time()-t0:.0f}s")
        if va>best_val: best_val=va; best_w=copy.deepcopy(model.state_dict())

    model.load_state_dict(best_w)
    acc,prec,rec,f1,auc=evaluate(model,te_l)
    print(f"\n  ✔ TEST Acc:{acc} Prec:{prec} Rec:{rec} F1:{f1} AUC:{auc}")

    ckpt_name  = mname.replace('-','_')+'_best.pth'
    ckpt_local = os.path.join(SAVE_DIR,ckpt_name)
    torch.save(model.state_dict(),ckpt_local)
    save_to_drive(ckpt_name)

    p,s,fl,im=compute_efficiency(model)
    table1[mname]={'Accuracy (%)':acc,'Precision (%)':prec,
                   'Recall (%)':rec,  'F1-Score (%)':f1,'AUC (%)':auc}
    table3[mname]={'Parameters (M)':p,'Model Size (MB)':s,
                   'FLOPs (G)':fl,    'Inference Time (ms)':im,'Accuracy (%)':acc}
    if acc>best_acc: best_acc=acc; best_backbone=mname

    with open(results_json_local,'w') as f:
        json.dump({'table1':table1,'table3':table3,
                   'best_backbone':best_backbone},f,indent=2)
    pd.DataFrame(table1).T.to_csv(os.path.join(SAVE_DIR,'table1.csv'))
    pd.DataFrame(table3).T.to_csv(os.path.join(SAVE_DIR,'table3.csv'))
    save_to_drive('all_results.json')
    save_to_drive('table1.csv')
    save_to_drive('table3.csv')
    print(f"  ✅ {mname} done! Completed:{list(table1.keys())}")

    del model,tr_l,vl_l,te_l
    torch.cuda.empty_cache(); gc.collect()

# ── Table 2: Deep Features + Classifiers ───────
print(f"\n{'='*55}\n  TABLE 2 — Features from: {best_backbone}\n{'='*55}")

ckpt_local = os.path.join(SAVE_DIR,best_backbone.replace('-','_')+'_best.pth')
ckpt_drive = os.path.join(DRIVE_DIR,best_backbone.replace('-','_')+'_best.pth')
if not os.path.exists(ckpt_local) and os.path.exists(ckpt_drive):
    shutil.copy2(ckpt_drive,ckpt_local)

model=get_model(best_backbone)
model.load_state_dict(torch.load(ckpt_local,map_location=DEVICE))
model=model.to(DEVICE).eval()
for p in model.parameters(): p.requires_grad=False

captured=[]
def hook(m,i,o): captured.append(o.detach().cpu())

n=best_backbone.lower().replace('-','').replace('_','')
if 'alexnet' in n or 'vgg' in n:
    handle=model.classifier[-2].register_forward_hook(hook)
elif 'resnet' in n:
    handle=model.avgpool.register_forward_hook(hook)
elif 'densenet' in n:
    handle=model.features.register_forward_hook(hook)
elif 'efficientnet' in n:
    handle=model.classifier.register_forward_hook(hook)
else:
    handle=list(model.children())[-2].register_forward_hook(hook)

def collect(loader):
    F,L=[],[]
    with torch.no_grad():
        for imgs,labels in loader:
            captured.clear()
            model(imgs.to(DEVICE))
            f=captured[0].view(captured[0].size(0),-1)
            F.append(f.numpy()); L.extend(labels.numpy())
    return np.vstack(F),np.array(L)

bs_feat   = BATCH_MAP.get(best_backbone,32)
tr_l,_,te_l = make_loaders(bs_feat)
X_tr,y_tr = collect(tr_l)
X_te,y_te = collect(te_l)
handle.remove(); del model,tr_l,te_l
torch.cuda.empty_cache(); gc.collect()
print(f"Features — Train:{X_tr.shape} Test:{X_te.shape}")

classifiers={
    'Logistic Regression': LogisticRegression(max_iter=1000,n_jobs=-1),
    'Decision Tree':       DecisionTreeClassifier(max_depth=15,random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200,n_jobs=-1,random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5,n_jobs=-1),
    'Linear SVM':          SVC(kernel='linear',probability=True,max_iter=2000),
    'RBF-SVM':             SVC(kernel='rbf',probability=True,C=10,gamma='scale'),
    'XGBoost':             XGBClassifier(n_estimators=200,learning_rate=0.1,
                                          eval_metric='mlogloss',n_jobs=-1,random_state=42),
}
table2={}
for cn,clf in classifiers.items():
    print(f"  {cn}...",end=' ',flush=True)
    t0=time.time(); clf.fit(X_tr,y_tr); print(f"{time.time()-t0:.1f}s")
    preds=clf.predict(X_te)
    acc =accuracy_score(y_te,preds)*100
    prec=precision_score(y_te,preds,average='weighted',zero_division=0)*100
    rec =recall_score(y_te,preds,   average='weighted',zero_division=0)*100
    f1  =f1_score(y_te,preds,        average='weighted',zero_division=0)*100
    try:
        prob =clf.predict_proba(X_te) if hasattr(clf,'predict_proba') \
              else np.eye(NUM_CLASSES)[preds]
        ybin =label_binarize(y_te,classes=list(range(NUM_CLASSES)))
        auc  =roc_auc_score(ybin,prob,multi_class='ovr',average='weighted')*100
    except: auc=0.0
    table2[cn]={'Feature Extractor':best_backbone,'Classifier':cn,
                'Accuracy (%)':round(acc,2),'Precision (%)':round(prec,2),
                'Recall (%)':round(rec,2),'F1-Score (%)':round(f1,2),
                'AUC (%)':round(auc,2)}
    print(f"    Acc:{round(acc,2)} Prec:{round(prec,2)} "
          f"Rec:{round(rec,2)} F1:{round(f1,2)} AUC:{round(auc,2)}")

pd.DataFrame(table2).T.to_csv(os.path.join(SAVE_DIR,'table2.csv'))
save_to_drive('table2.csv')

# ── Print ALL 3 Tables ──────────────────────────
S="─"*76
print(f"\n\n{'═'*76}")
print("  TABLE 1 — Transfer Learning Model Comparison")
print(f"{'═'*76}")
print(f"{'Model':<18}{'Acc%':>9}{'Prec%':>9}{'Rec%':>9}{'F1%':>9}{'AUC%':>9}")
print(S)
for m,r in table1.items():
    print(f"{m:<18}{r['Accuracy (%)']:>9}{r['Precision (%)']:>9}"
          f"{r['Recall (%)']:>9}{r['F1-Score (%)']:>9}{r['AUC (%)']:>9}")

print(f"\n{'═'*76}")
print("  TABLE 2 — Deep Features + Classical Classifiers")
print(f"{'═'*76}")
print(f"{'Classifier':<25}{'Acc%':>9}{'Prec%':>9}{'Rec%':>9}{'F1%':>9}{'AUC%':>9}")
print(S)
for c,r in table2.items():
    print(f"{c:<25}{r['Accuracy (%)']:>9}{r['Precision (%)']:>9}"
          f"{r['Recall (%)']:>9}{r['F1-Score (%)']:>9}{r['AUC (%)']:>9}")

print(f"\n{'═'*76}")
print("  TABLE 3 — Computational Efficiency")
print(f"{'═'*76}")
print(f"{'Model':<18}{'Params(M)':>11}{'Size(MB)':>10}"
      f"{'FLOPs(G)':>10}{'Infer(ms)':>11}{'Acc%':>9}")
print(S)
for m,r in table3.items():
    print(f"{m:<18}{r['Parameters (M)']:>11}{r['Model Size (MB)']:>10}"
          f"{str(r['FLOPs (G)']):>10}{r['Inference Time (ms)']:>11}"
          f"{r['Accuracy (%)']:>9}")

# Final save
with open(results_json_local,'w') as f:
    json.dump({'table1':table1,'table2':table2,
               'table3':table3,'best_backbone':best_backbone},f,indent=2)
for fname in ['all_results.json','table1.csv','table2.csv','table3.csv']:
    save_to_drive(fname)

print(f"\n🏆 Best model: {best_backbone} ({best_acc}%)")
print("🎉 ALL DONE! Copy numbers into Task_01.docx")

✅ Loaded from Drive. Done: ['AlexNet', 'VGG16', 'VGG19', 'ResNet18', 'ResNet50', 'ResNet101', 'DenseNet121']
  📥 Restored VGG16_best.pth
  📥 Restored VGG19_best.pth
  📥 Restored ResNet18_best.pth
  📥 Restored ResNet50_best.pth
  📥 Restored ResNet101_best.pth
  📥 Restored DenseNet121_best.pth

Status: 7/8 done | Best: ResNet50 (80.25%)
Still need: ['EfficientNet-B0']
⏭️  AlexNet            Acc:67.54%
⏭️  VGG16              Acc:68.88%
⏭️  VGG19              Acc:70.75%
⏭️  ResNet18           Acc:77.98%
⏭️  ResNet50           Acc:80.25%
⏭️  ResNet101          Acc:77.04%
⏭️  DenseNet121        Acc:76.17%

  EfficientNet-B0  batch=32


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

  EfficientNet-B0    trainable: 3,295,695/ 4,016,515 (82.1%)

  📌 Phase 1 (4 ep | LR=0.001)
  Ep1/4 Loss:1.4518 Train:62.5% Val:71.0% 87s
  Ep2/4 Loss:0.9185 Train:78.9% Val:72.7% 69s
  Ep3/4 Loss:0.8010 Train:85.4% Val:74.9% 67s
  Ep4/4 Loss:0.7290 Train:88.5% Val:76.9% 73s

  🔓 Phase 2 (6 ep | LR=1e-05)
  Ep1/6 Loss:0.7056 Train:89.1% Val:78.5% 83s
  Ep2/6 Loss:0.6813 Train:91.5% Val:77.9% 73s
  Ep3/6 Loss:0.6873 Train:90.4% Val:77.8% 75s
  Ep4/6 Loss:0.6779 Train:91.3% Val:78.1% 78s
  Ep5/6 Loss:0.6770 Train:91.4% Val:79.8% 69s
  Ep6/6 Loss:0.6756 Train:91.6% Val:77.8% 70s

  ✔ TEST Acc:77.91 Prec:82.66 Rec:77.91 F1:79.64 AUC:94.05
  ☁️  EfficientNet_B0_best.pth
  ☁️  all_results.json
  ☁️  table1.csv
  ☁️  table3.csv
  ✅ EfficientNet-B0 done! Completed:['AlexNet', 'VGG16', 'VGG19', 'ResNet18', 'ResNet50', 'ResNet101', 'DenseNet121', 'EfficientNet-B0']

  TABLE 2 — Features from: ResNet50
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/

100%|██████████| 97.8M/97.8M [00:00<00:00, 166MB/s]


  ResNet50           trainable:22,077,447/23,522,375 (93.9%)
Features — Train:(4109, 2048) Test:(2241, 2048)
  Logistic Regression... 8.3s
    Acc:78.54 Prec:83.99 Rec:78.54 F1:80.38 AUC:94.26
  Decision Tree... 13.8s
    Acc:69.08 Prec:80.46 Rec:69.08 F1:72.54 AUC:77.24
  Random Forest... 30.2s
    Acc:78.94 Prec:86.8 Rec:78.94 F1:81.33 AUC:95.84
  K-Nearest Neighbors... 0.0s
    Acc:75.15 Prec:84.34 Rec:75.15 F1:77.97 AUC:92.58
  Linear SVM... 24.6s
    Acc:77.6 Prec:82.96 Rec:77.6 F1:79.43 AUC:94.53
  RBF-SVM... 42.5s
    Acc:78.8 Prec:84.7 Rec:78.8 F1:80.76 AUC:94.2
  XGBoost... 626.7s
    Acc:80.14 Prec:86.6 Rec:80.14 F1:82.13 AUC:95.96
  ☁️  table2.csv


════════════════════════════════════════════════════════════════════════════
  TABLE 1 — Transfer Learning Model Comparison
════════════════════════════════════════════════════════════════════════════
Model                  Acc%    Prec%     Rec%      F1%     AUC%
──────────────────────────────────────────────────────────────────